In [20]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torch.optim as optim         
import os
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import warnings
import sys
from PIL import Image

现在对$MobileNet-V2$进行复现

In [21]:
class InvertedResidualBlock(nn.Module):
    '''带有残差连接的倒残差块，其中使用了深度可分离卷积架构'''
    def __init__(self , in_channels , out_channels , stride , expand_ratio):
        super(InvertedResidualBlock , self).__init__()
        self.layer1 = nn.Conv2d(in_channels , in_channels * expand_ratio , kernel_size = 1 ,  stride = 1 , padding = 0 ,bias = False)
        self.layer2 = nn.BatchNorm2d(in_channels * expand_ratio)
        self.layer3 = nn.ReLU6(inplace = True)

        self.layer4 = nn.Conv2d(in_channels * expand_ratio , in_channels*expand_ratio , kernel_size=3 , stride=stride , padding = 1 , groups=in_channels*expand_ratio , bias = False)
        self.layer5 = nn.BatchNorm2d(in_channels * expand_ratio)
        self.layer6 = nn.ReLU6(inplace = True)

        self.layer7 = nn.Conv2d(in_channels*expand_ratio , out_channels , kernel_size=1 , stride=1 , padding=0 , bias = False )
        self.layer8 = nn.BatchNorm2d(out_channels)

        self.downsample = None
        if stride == 1 and in_channels == out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels , out_channels , kernel_size=1 , stride=stride , padding=0 , bias = False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        identity = x
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.layer5(out)
        out = self.layer6(out)
        out = self.layer7(out)
        out = self.layer8(out)
        if self.downsample is not None:
            identity = self.downsample(x)
            out += identity
        return out

In [22]:
x = torch.randn(1,3,112,112)
model = InvertedResidualBlock(3,64,1,6)
out = model(x)
print(out.shape)

torch.Size([1, 64, 112, 112])


In [23]:
class MobileNetV2(nn.Module):
    def __init__(self , in_channels = 3 , num_classes = 10):
        super(MobileNetV2 , self).__init__()

        self.pre_layer = nn.Sequential(
            nn.Conv2d(in_channels , 32 , kernel_size = 3 , stride=2 , padding = 1 , bias = False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace = True)
        )

        self.layer1 = nn.Sequential(
            InvertedResidualBlock(32 , 16 , 1 , expand_ratio=1),

            InvertedResidualBlock(16 , 24 , 2 , expand_ratio=6),
            InvertedResidualBlock(24 , 24 , 1 , expand_ratio=6),

            InvertedResidualBlock(24 , 32 , 2 , expand_ratio=6),
            InvertedResidualBlock(32 , 32 , 1 , expand_ratio=6),
            InvertedResidualBlock(32 , 32 , 1 , expand_ratio=6),

            InvertedResidualBlock(32 , 64 , 2 , expand_ratio=6),
            InvertedResidualBlock(64 , 64 , 1 , expand_ratio=6),    
            InvertedResidualBlock(64 , 64 , 1 , expand_ratio=6),
            InvertedResidualBlock(64 , 64 , 1 , expand_ratio=6),

            InvertedResidualBlock(64 , 96 , 1 , expand_ratio=6),
            InvertedResidualBlock(96 , 96 , 1 , expand_ratio=6),    
            InvertedResidualBlock(96 , 96 , 1 , expand_ratio=6),

            InvertedResidualBlock(96 , 160 , 2 , expand_ratio=6),
            InvertedResidualBlock(160 , 160 , 1 , expand_ratio=6),
            InvertedResidualBlock(160 , 160 , 1 , expand_ratio=6),

            InvertedResidualBlock(160 , 320 , 1 , expand_ratio=6),
        )

        self.last_layer = nn.Sequential(
            nn.Conv2d(320 , 1280 , kernel_size=1 , stride=1 , padding=0 , bias = False),
            nn.BatchNorm2d(1280),
            nn.ReLU6(inplace = True),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.classifier = nn.Linear(1280 , num_classes)
    def forward(self, x):
        out = self.pre_layer(x)
        out = self.layer1(out)
        out = self.last_layer(out)
        out = out.view(out.size(0) , -1)
        out = self.classifier(out)
        return out
    
# x = torch.randn(30,3,224,224)
# model = MobileNetV2(in_channels=3 , num_classes=10)
# out = model(x)
# print(out.shape)

In [28]:
def initialize_weights(model):
    for m in model.modules():
        if isinstance(m , nn.Conv2d):
            nn.init.kaiming_normal_(m.weight , mode='fan_out' , nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias , 0)
        elif isinstance(m , nn.BatchNorm2d):
            nn.init.constant_(m.weight , 1)
            nn.init.constant_(m.bias , 0)
        elif isinstance(m , nn.Linear):
            nn.init.normal_(m.weight , 0 , 0.01)
            nn.init.constant_(m.bias , 0)

In [29]:
def plot(train_losses, test_losses, train_accs, test_accs):
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(train_losses , label='Train Loss' , color='blue' , linewidth=5)
    plt.plot(test_losses , label='Test Loss' , color='red' , linewidth=5)
    plt.title('Loss over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(train_accs , label='Train Accuracy' , color='blue' , linewidth=5)
    plt.plot(test_accs , label='Test Accuracy' , color='red' , linewidth=5)
    plt.title('Accuracy over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    
    plt.show()

In [32]:
def train():
    BATCH_SIZE = 32
    EPOCHS = 2
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device: ", device)

    model = MobileNetV2(in_channels=3 , num_classes=10).to(device)
    model.apply(initialize_weights)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters() , lr=0.001)

    print("prepare data...")

    transform = transforms.Compose([   
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])

    train_dataset = datasets.CIFAR10(root='E:\mastercode\data' , train=True , transform=transform , download=True)
    test_dataset = datasets.CIFAR10(root='E:\mastercode\data' , train=False , transform=transform , download=True)

    train_loader = DataLoader(train_dataset , batch_size=BATCH_SIZE , shuffle=True , num_workers=2)
    test_loader = DataLoader(test_dataset , batch_size=BATCH_SIZE , shuffle=False , num_workers=2)

    print("start training...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        loop = tqdm(train_loader , file=sys.stdout)
        for inputs , labels in loop:
            inputs , labels = inputs.to(device) , labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs , labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            loop.set_description(f'Epoch [{epoch+1}/{EPOCHS}]')
            loop.set_postfix(loss=loss.item() , acc=100.*correct/total)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = 100. * correct / total
        train_losses.append(epoch_loss)
        train_accs.append(epoch_acc)

        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs , labels in test_loader:
                inputs , labels = inputs.to(device) , labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs , labels)

                test_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        epoch_test_loss = test_loss / len(test_loader.dataset)
        epoch_test_acc = 100. * correct / total
        test_losses.append(epoch_test_loss)
        test_accs.append(epoch_test_acc)

        print(f'Epoch [{epoch+1}/{EPOCHS}] Train Loss: {epoch_loss:.4f} Train Acc: {epoch_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} Test Acc: {epoch_test_acc:.2f}%')
    plot(train_losses, test_losses, train_accs, test_accs)

In [ ]:
train()

Using device:  cuda
prepare data...
Files already downloaded and verified
Files already downloaded and verified
start training...
Epoch [1/2]:  18%|█▊        | 276/1563 [00:25<01:45, 12.22it/s, acc=15.2, loss=2.05]